# Сборка единого датасета под Задачу 1 (табличные данные)

Третий ноутбук конвейера. Из нарезанного среза (`data/processed/*.parquet`) собираем **один плоский
датасет** для рекомендательной полносвязной сети: предсказание оценки `stars`, которую пользователь
поставит заведению, по табличным признакам пользователя и заведения.

**Логика:** «позвоночник» — таблица `reviews` (1 строка = 1 отзыв = 1 пара пользователь×заведение
с известным таргетом). К ней слева подклеиваем признаки `business` (по `business_id`) и `users`
(по `user_id`).

```
reviews ──LEFT JOIN business ON business_id──┐
        └─LEFT JOIN users    ON user_id──────┴──► task1_dataset (target = stars)
```

Текст отзывов и таблица `tips` сюда НЕ входят — это сырьё Задачи 2.

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

_PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(_PROJECT))
from _constants import REVIEWS_PARQUET, BUSINESS_PARQUET, USERS_PARQUET, PROCESSED

TOP_K_CATEGORIES = 20          # сколько самых частых категорий заведений кодируем multi-hot
OUT = PROCESSED / "task1_dataset.parquet"
SEED = 42

## 1. Загрузка таблиц среза

Из `reviews` берём только нужные колонки (текст и голоса отзыва не грузим — см. раздел 3 про
утечки). Сразу переименовываем колонки, которые иначе столкнулись бы при join
(`stars`, `review_count`, `useful/funny/cool` есть в нескольких таблицах).

In [2]:
# reviews: ключи + таргет + дата (без text/useful/funny/cool)
reviews = pd.read_parquet(
    REVIEWS_PARQUET, columns=["review_id", "user_id", "business_id", "stars", "date"])

# business: убираем чисто справочные поля (name, postal_code, сырые city/state)
business = pd.read_parquet(BUSINESS_PARQUET).drop(
    columns=["name", "postal_code", "city", "state"]).rename(
    columns={"stars": "biz_avg_stars", "review_count": "biz_review_count"})

# users: разводим имена, которые столкнулись бы с business/reviews
users = pd.read_parquet(USERS_PARQUET).rename(columns={
    "review_count": "user_review_count",
    "useful": "user_useful", "funny": "user_funny", "cool": "user_cool"})

print("reviews :", reviews.shape)
print("business:", business.shape)
print("users   :", users.shape)

reviews : (664751, 5)
business: (17556, 9)
users   : (209093, 10)


## 2. Объединение в один датасет

`reviews` — левая таблица, гранулярность сохраняется (1 строка = 1 отзыв).

In [3]:
df = (reviews
      .merge(business, on="business_id", how="left")
      .merge(users, on="user_id", how="left"))
print("Объединённый датасет:", df.shape)
assert len(df) == len(reviews), "join размножил строки — проверь ключи"

# 5 user_id из reviews отсутствуют в users.parquet (выпали на препроцессинге) ->
# у них нет профиля. Таких строк единицы — отбрасываем.
before = len(df)
df = df.dropna(subset=["average_stars"]).reset_index(drop=True)
print(f"Удалено строк без профиля пользователя: {before - len(df)}")
df.head(3)

Объединённый датасет: (664751, 22)
Удалено строк без профиля пользователя: 6


,review_id,user_id,business_id,stars,date,city_state,latitude,longitude,biz_avg_stars,biz_review_count,...,price_range,user_review_count,yelping_since,user_useful,user_funny,user_cool,fans,average_stars,n_friends,n_elite_years
0,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3.0,2014-02-05 20:30:30,"Tucson, AZ",32.207233,-110.980864,3.5,47,...,1.0,1332.0,2012-09-04 23:57:25,1660.0,675.0,1300.0,58.0,4.69,119.0,9.0
1,UBp0zWyH60Hmw6Fsasei7w,4Uh27DgGzsp6PqrH913giQ,otQS34_MymijPTdNBoBdCw,4.0,2011-10-27 17:12:05,"Tucson, AZ",32.255834,-110.960560,4.0,492,...,2.0,50.0,2009-09-29 22:24:04,146.0,106.0,77.0,2.0,4.27,40.0,2.0
2,rV6AWGN4rYORMQY8dVP41g,dKoIp8vsKFH4cbmGSYy2IQ,0ICfbEImE0gUZc4kSZ7QHg,5.0,2013-11-14 04:58:09,"Edmonton, AB",53.499624,-113.456746,2.5,27,...,NaN,11.0,2012-02-20 04:58:56,15.0,8.0,3.0,0.0,4.64,0.0,0.0


## 3. Чистка утечек и инжиниринг признаков

**Утечки таргета** уже исключены тем, что мы не грузили `text` и `review.useful/funny/cool`
(они известны только ПОСЛЕ написания отзыва). Дополнительно:

- `account_age_days` — стаж пользователя на момент отзыва (`date − yelping_since`);
- `price_range` пуст у ~44% заведений → флаг `price_known` + импутация медианой;
- `user_avg_minus_biz` — насколько пользователь в среднем строже/добрее среднего по заведению.

> Замечание: `biz_avg_stars`, `biz_review_count`, `average_stars` — агрегаты по всем отзывам
> (мягкая утечка). Поэтому ниже делаем **временной** train/val/test-сплит по дате.

In [4]:
# Возраст аккаунта на момент отзыва (в днях)
df["date"] = pd.to_datetime(df["date"])
df["yelping_since"] = pd.to_datetime(df["yelping_since"])
df["account_age_days"] = (df["date"] - df["yelping_since"]).dt.days.clip(lower=0)

# Ценовой диапазон: флаг известности + импутация
df["price_known"] = df["price_range"].notna().astype(int)
df["price_range"] = df["price_range"].fillna(df["price_range"].median())

# Производный признак: склонность пользователя относительно среднего по заведению
df["user_avg_minus_biz"] = df["average_stars"] - df["biz_avg_stars"]

df = df.drop(columns=["yelping_since"])

## 4. Кодирование категориальных признаков

- `categories` — мульти-метка: одно заведение принадлежит нескольким категориям. Кодируем
  **multi-hot** по топ-K самых частых категорий среза (`cat_*`).
- `city_state` — всего 3 города → one-hot (`city_*`).

In [5]:
# --- multi-hot по топ-K категорий ---
cat_sets = df["categories"].fillna("").apply(
    lambda s: {x.strip() for x in s.split(",") if x.strip()})
top_cats = (df["categories"].dropna().str.split(", ").explode()
            .value_counts().head(TOP_K_CATEGORIES).index.tolist())

def slug(name):
    return "cat_" + re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")

for cat in top_cats:
    df[slug(cat)] = cat_sets.apply(lambda st: int(cat in st))
cat_cols = [slug(c) for c in top_cats]

# --- one-hot по городу ---
city_oh = pd.get_dummies(df["city_state"], prefix="city").astype(int)
df = pd.concat([df, city_oh], axis=1)
city_cols = city_oh.columns.tolist()

df = df.drop(columns=["categories", "city_state"])
print(f"Категорий multi-hot: {len(cat_cols)} | городов one-hot: {len(city_cols)}")
print("Топ-категории:", top_cats[:8], "...")

Категорий multi-hot: 20 | городов one-hot: 3
Топ-категории: ['Restaurants', 'Food', 'Nightlife', 'Bars', 'American (Traditional)', 'Breakfast & Brunch', 'American (New)', 'Mexican'] ...


## 5. Временной train/val/test-сплит

Делим по `date` (70% / 15% / 15%): учимся на прошлом, проверяем на будущем — так агрегатные
признаки не «подсматривают» будущее.

In [6]:
df = df.sort_values("date").reset_index(drop=True)
q70, q85 = df["date"].quantile([0.70, 0.85])
df["split"] = np.where(df["date"] <= q70, "train",
              np.where(df["date"] <= q85, "val", "test"))
print(df["split"].value_counts(normalize=True).round(3).to_dict())
print("границы:", q70.date(), q85.date())

{'train': 0.7, 'val': 0.15, 'test': 0.15}
границы: 2019-01-18 2020-04-23


## 6. Сохранение готового датасета

Итог — один parquet: ключи (для справки), `date`, `split`, таргет `stars` и табличные признаки.

In [7]:
PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUT, index=False)

id_cols = ["review_id", "user_id", "business_id"]
num_cols = ["biz_avg_stars", "biz_review_count", "is_open", "price_range", "price_known",
            "latitude", "longitude", "user_review_count", "average_stars",
            "user_useful", "user_funny", "user_cool", "fans", "n_friends",
            "n_elite_years", "account_age_days", "user_avg_minus_biz"]
feature_cols = num_cols + cat_cols + city_cols

print(f"[ok] {OUT.name}: {df.shape[0]:,} строк, {df.shape[1]} колонок")
print(f"  таргет: stars | признаков: {len(feature_cols)} "
      f"({len(num_cols)} числовых + {len(cat_cols)} категорий + {len(city_cols)} городов)")
print(f"  служебные (не признаки): {id_cols + ['date', 'split']}")
print("\nПризнаки:")
for col in feature_cols:
    print("  -", col)

[ok] task1_dataset.parquet: 664,745 строк, 46 колонок
  таргет: stars | признаков: 40 (17 числовых + 20 категорий + 3 городов)
  служебные (не признаки): ['review_id', 'user_id', 'business_id', 'date', 'split']

Признаки:
  - biz_avg_stars
  - biz_review_count
  - is_open
  - price_range
  - price_known
  - latitude
  - longitude
  - user_review_count
  - average_stars
  - user_useful
  - user_funny
  - user_cool
  - fans
  - n_friends
  - n_elite_years
  - account_age_days
  - user_avg_minus_biz
  - cat_restaurants
  - cat_food
  - cat_nightlife
  - cat_bars
  - cat_american_traditional
  - cat_breakfast_brunch
  - cat_american_new
  - cat_mexican
  - cat_shopping
  - cat_sandwiches
  - cat_event_planning_services
  - cat_pizza
  - cat_beauty_spas
  - cat_burgers
  - cat_coffee_tea
  - cat_italian
  - cat_cocktail_bars
  - cat_seafood
  - cat_salad
  - cat_home_services
  - city_Edmonton, AB
  - city_St Petersburg, FL
  - city_Tucson, AZ
